# arcroad — Phase 2e: explaining the flagged third (v1.0)

**What v0.9 left unexplained.** 12,585 points routed; 4,284 of them — **34%** — came out
above a detour index of 50 and were flagged rather than understood. A method that cannot
account for a third of what it measures is not finished, however good its headline looks.

v1.0 does not add a new metric. It interrogates the one we have:

1. **What ARE the flagged points?** Broken down by ring band, road class, distance from
   the study-area edge, and whether they cluster. Each breakdown tests a different
   explanation.
2. **Edge effects (E-1).** Points near the boundary of the measured area have fewer
   roads around them in the graph, so their detours inflate for a reason that has
   nothing to do with the real road. This is tested directly and, if real, excluded.
3. **Distance-stratified sectors (E-2).** v0.9's sectors sample at very different
   depths — 7.6 km to 23.4 km from the centre. Comparing them within ring bands removes
   that difference entirely, so a sector cannot look bad merely for sampling further out.
4. **Sample adequacy (E-3).** Every sector reports whether its sample is big enough to
   carry the claim being made about it.
5. **Version string (F-14).** Fixed properly this time: the earlier patch missed because
   it searched for an escaped dash where the file holds a literal one.

**Status:** v1.0, 25 Oct 2026. Same measurement engine, unchanged. Everything new here
is diagnosis of its output.

## 1. Install

In [ ]:
!pip -q install osmium tabulate
!apt-get -qq install -y osmium-tool > /dev/null 2>&1
import osmium, pandas, numpy, scipy
print("ready | pandas", pandas.__version__, "| scipy", scipy.__version__)

## 2. Configuration

In [ ]:
EXTRACT_URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
PBF_PATH    = "/content/southern-zone.osm.pbf"
CLIP_PATH   = "/content/hyderabad_buffered.osm.pbf"
CLUSTER_M     = 150       # F-11: de-duplicate worst-N within this radius
FLAG_CLUSTER_M= 250       # F-13: cluster radius for the flagged population
OUT_DIR     = "/content/hma_out"

ZONES = {
    "core_city":  (78.380, 17.370, 78.520, 17.460),
    "patancheru": (78.230, 17.490, 78.330, 17.570),
    "hmda_all":   (78.050, 17.050, 78.850, 17.750),
}
CLIP_BUFFER_DEG = 0.20      # F-3: clip wider than we measure (~22 km of margin)

DRIVABLE = {
    "motorway","trunk","primary","secondary","tertiary",
    "motorway_link","trunk_link","primary_link","secondary_link","tertiary_link",
    "unclassified","residential","living_street","service","road",
}
ARTERIAL = {"motorway","trunk","primary","secondary"}

CROSS_MIN_M   = 12
CROSS_MAX_M   = 70
BEARING_TOL   = 35        # F-2: degrees from antiparallel still accepted
SAMPLE_STRIDE = 4         # uniform across the city — every sector sampled alike
FOCUS_ZONE    = None      # v0.8 measures every named area equally
FOCUS_STRIDE  = 5
MAX_SAMPLES   = 16000
PLACE_MAX_M   = 3000      # a point further than this from any named place is unassigned
MIN_AREA_N    = 30        # below this, an area's median is noise

# --- the ring frame ---------------------------------------------------------
CITY = (78.4867, 17.3850)   # Charminar-ish; the centre every bearing is taken from
ORR_INNER_KM  = 14.0        # inside this, the core (ring radius 16.6-28.8 km)
ORR_BAND_KM   = 26.0        # between: the ORR band itself
# Nineteen sectors named for the exit on that bearing (0 deg = north, clockwise).
# The SECTOR is geometry we compute; the name is a label, and published exit lists
# disagree with one another, so names are indicative until derived from OSM junctions.
# Bearings computed from each interchange's approximate position, not spaced evenly:
# real gaps run from 7 to 29 degrees, and ring radius from 16.6 to 28.8 km. Evenly
# spaced sectors put Patancheru in the wrong wedge, so the geography is used instead.
ORR_EXITS = [
    ("Dundigal",        339), ("Medchal",         359),
    ("Shamirpet",        20), ("Keesara",          41),
    ("Ghatkesar",        70), ("Taramatipet",      92),
    ("Pedda Amberpet",  114), ("Bongulur",        138),
    ("Raviryal",        159), ("Tukkuguda",       177),
    ("Pedda Golconda",  194), ("Shamshabad",      212),
    ("Rajendranagar",   233), ("TSPA",            255),
    ("Narsingi",        271), ("Gandipet",        284),
    ("Edulanagulapally",297), ("Patancheru",      304),
    ("Sultanpur",       322),
]
ROUTE_LIMIT_M = 25000     # components proved clean in v0.7; 25 km is ample
DI_SEVERE     = 5.0
DI_REVIEW     = 50.0      # F-6: a FLAG on points, never a filter on the headline

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("configured | buffer", CLIP_BUFFER_DEG, "deg | bearing tol", BEARING_TOL, "deg")

## 3. Download and clip with a buffer (F-3)

In [ ]:
import subprocess, os

if not (os.path.exists(CLIP_PATH) and os.path.getsize(CLIP_PATH) > 100_000):
    if not (os.path.exists(PBF_PATH) and os.path.getsize(PBF_PATH) > 1_000_000):
        print("downloading", EXTRACT_URL)
        if subprocess.call(["wget","-q","--show-progress","-O",PBF_PATH,EXTRACT_URL]) != 0:
            raise SystemExit("Download failed — check EXTRACT_URL.")
    x0, y0, x1, y1 = ZONES["hmda_all"]
    b = CLIP_BUFFER_DEG
    bbox = f"{x0-b},{y0-b},{x1+b},{y1+b}"
    print("clipping to buffered box", bbox)
    r = subprocess.run(["osmium","extract","-b",bbox,PBF_PATH,"-o",CLIP_PATH,
                        "--overwrite","-s","complete_ways"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("Clip failed.")

print(round(os.path.getsize(CLIP_PATH)/1e6, 1), "MB clipped (buffered)")

## 4. Pure algorithm (unit-tested in §8)

All plain maths, no OSM dependency, so the method can be checked against cases whose
answers are known by construction.

In [ ]:
import numpy as np, math

MILITARY_KEYS = {"military", "military_service"}
MILITARY_LANDUSE = {"military"}
MILITARY_VALUES = {"danger_area","barracks","naval_base","airfield","training_area"}

def strip_military(tags):
    hit = (any(k in MILITARY_KEYS for k in tags)
           or tags.get("landuse") in MILITARY_LANDUSE
           or tags.get("military") in MILITARY_VALUES)
    if not hit:
        return tags, False
    clean = {k: v for k, v in tags.items()
             if k not in MILITARY_KEYS and k not in ("landuse","name","operator","description")}
    clean["access_restricted"] = "yes"
    return clean, True

def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin((p2-p1)/2)**2
         + math.cos(p1)*math.cos(p2)*math.sin(math.radians(lon2-lon1)/2)**2)
    return 2 * R * math.asin(math.sqrt(a))

def bearing_deg(lon1, lat1, lon2, lat2):
    """Compass bearing from point 1 to point 2, 0-360."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    y = math.sin(dl) * math.cos(p2)
    x = math.cos(p1)*math.sin(p2) - math.sin(p1)*math.cos(p2)*math.cos(dl)
    return (math.degrees(math.atan2(y, x)) + 360.0) % 360.0

def angle_between(b1, b2):
    """Smallest absolute difference between two bearings, 0-180."""
    d = abs(b1 - b2) % 360.0
    return d if d <= 180.0 else 360.0 - d

def is_antiparallel(b1, b2, tol):
    """F-2: true when the two headings are opposite within tol degrees."""
    return abs(angle_between(b1, b2) - 180.0) <= tol

def way_level(tags):
    """F-1: the vertical level this way occupies. Explicit layer wins; otherwise a
    bridge is above and a tunnel below the surface."""
    raw = tags.get("layer")
    if raw is not None:
        try:
            return int(float(raw))
        except (TypeError, ValueError):
            pass
    if tags.get("bridge") not in (None, "no"):
        return 1
    if tags.get("tunnel") not in (None, "no"):
        return -1
    return 0

def levels_compatible(levels_a, levels_b):
    """Two points can only be 'across the road' if they share a level."""
    return bool(set(levels_a) & set(levels_b))

def oneway_dirs(tags):
    ow = tags.get("oneway")
    if ow in ("yes","true","1"):
        return True, False
    if ow == "-1":
        return False, True
    if tags.get("junction") == "roundabout" and ow is None:
        return True, False
    return True, True

def zone_of(lon, lat, zones):
    return [n for n,(x0,y0,x1,y1) in zones.items() if x0 <= lon <= x1 and y0 <= lat <= y1]

def detour_index(network_m, straight_m):
    if straight_m is None or straight_m <= 0:
        return None
    if network_m is None or not np.isfinite(network_m):
        return None
    return network_m / straight_m

print("algorithm loaded")

## 5. Parse the road network

Now also records each way's level and the bearing of every segment.

In [ ]:
import osmium, collections

class GraphPass(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.ways = []
        self.coords = {}
        self.places = []
        self.military_stripped = 0

    def way(self, w):
        tags = dict(w.tags)
        hw = tags.get("highway")
        if hw not in DRIVABLE:
            return
        tags, stripped = strip_military(tags)
        if stripped:
            self.military_stripped += 1
        try:
            nodes = [n.ref for n in w.nodes]
            for n in w.nodes:
                self.coords[n.ref] = (n.lon, n.lat)
        except Exception:
            return
        if len(nodes) < 2:
            return
        fwd, bwd = oneway_dirs(tags)
        divided = (tags.get("oneway") in ("yes","true","1","-1")
                   or tags.get("dual_carriageway") == "yes")
        self.ways.append((w.id, nodes, fwd, bwd, hw, divided, way_level(tags)))

    def node(self, n):
        """v0.8: every named place becomes a candidate area label."""
        t = dict(n.tags)
        p = t.get("place")
        nm = t.get("name")
        if not p or not nm:
            return
        if p not in ("city","town","suburb","neighbourhood","village","hamlet","quarter"):
            return
        try:
            self.places.append((nm, p, n.location.lon, n.location.lat))
        except Exception:
            pass

g = GraphPass()
g.apply_file(CLIP_PATH, locations=True)
print("drivable ways :", len(g.ways))
print("unique nodes  :", len(g.coords))
print("military strip:", g.military_stripped)
print("named places   :", len(g.places))

## 6. Build the graph, plus per-node levels and bearings

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

node_ids = np.fromiter(g.coords.keys(), dtype=np.int64)
idx_of = {nid: i for i, nid in enumerate(node_ids)}
lon = np.array([g.coords[n][0] for n in node_ids])
lat = np.array([g.coords[n][1] for n in node_ids])
N = len(node_ids)

rows, cols, vals = [], [], []
node_way     = collections.defaultdict(set)
node_levels  = collections.defaultdict(set)   # F-1
node_bearing = collections.defaultdict(list)  # F-2

for wid, nodes, fwd, bwd, hw, divided, lvl in g.ways:
    for a, b in zip(nodes[:-1], nodes[1:]):
        if a not in idx_of or b not in idx_of:
            continue
        ia, ib = idx_of[a], idx_of[b]
        d = haversine_m(lon[ia], lat[ia], lon[ib], lat[ib])
        if d <= 0:
            continue
        brg = bearing_deg(lon[ia], lat[ia], lon[ib], lat[ib])
        # travel bearing, so a oneway=-1 carriageway points the way you may drive
        node_bearing[a].append(brg if fwd else (brg + 180) % 360)
        node_bearing[b].append(brg if fwd else (brg + 180) % 360)
        if fwd:
            rows.append(ia); cols.append(ib); vals.append(d)
        if bwd:
            rows.append(ib); cols.append(ia); vals.append(d)
    for n in nodes:
        node_way[n].add(wid)
        node_levels[n].add(lvl)

G = csr_matrix((vals, (rows, cols)), shape=(N, N))
print(f"graph: {N:,} nodes, {G.nnz:,} directed edges")

## 6b. Connected components (F-15)

If a point and its target sit in different components of the road graph, there is no
route **in our data**. That is a data gap, not a detour. v0.6 counted these as enormous
detours, which is the prime suspect for the 19.5 km claims. They are now identified and
reported separately.

In [ ]:
from scipy.sparse.csgraph import connected_components

n_comp, comp_label = connected_components(G, directed=True, connection="weak")
sizes = np.bincount(comp_label)
order = np.argsort(sizes)[::-1]

print(f"weakly connected components: {n_comp:,}")
print(f"largest component          : {sizes[order[0]]:,} nodes "
      f"({100*sizes[order[0]]/N:.1f}% of the graph)")
if n_comp > 1:
    print(f"second largest             : {sizes[order[1]]:,} nodes")
print(f"nodes outside the largest  : {N - sizes[order[0]]:,}")
print()
print("A heavily fragmented graph means apparent severance is partly our data, not the")
print("road itself. Watch this before trusting any detour figure.")

## 7. Pair each sample with the far kerb — now with both filters

Rejection reasons are counted, so we can see how much of v0.4's result was artefact.

In [ ]:
from scipy.spatial import cKDTree

LAT0 = float(np.mean(lat))
xy = np.column_stack([lon * (111320.0*math.cos(math.radians(LAT0))), lat * 110540.0])
tree = cKDTree(xy)

cand = set()
for wid, nodes, fwd, bwd, hw, divided, lvl in g.ways:
    if hw in ARTERIAL and divided:
        cand.update(nodes)
cand = sorted(n for n in cand if n in idx_of)
print("nodes on divided arterials:", len(cand))

rej = collections.Counter()          # F-8: keyed by (zone, reason)
sources, targets, straight = [], [], []
seen = set()

def _zones_or_outside(nid_idx):
    z = zone_of(lon[nid_idx], lat[nid_idx], ZONES)
    return z if z else ["outside"]

def _note(nid_idx, reason):
    for z in _zones_or_outside(nid_idx):
        rej[(z, reason)] += 1

for k, nid in enumerate(cand):
    i = idx_of[nid]
    in_focus = bool(FOCUS_ZONE) and FOCUS_ZONE in zone_of(lon[i], lat[i], ZONES)
    stride = FOCUS_STRIDE if in_focus else SAMPLE_STRIDE
    if k % stride:
        continue
    if not node_bearing[nid]:
        _note(i, "no bearing")
        continue
    best, best_d = None, None
    for j in tree.query_ball_point(xy[i], CROSS_MAX_M):
        if j == i:
            continue
        njd = int(node_ids[j])
        d = haversine_m(lon[i], lat[i], lon[j], lat[j])
        if d < CROSS_MIN_M:
            continue
        if node_way[nid] & node_way[njd]:
            _note(i, "same way")
            continue
        if not levels_compatible(node_levels[nid], node_levels[njd]):
            _note(i, "grade separated")
            continue
        if not any(is_antiparallel(b1, b2, BEARING_TOL)
                   for b1 in node_bearing[nid] for b2 in node_bearing[njd]):
            _note(i, "not antiparallel")
            continue
        if best_d is None or d < best_d:
            best, best_d = j, d
    if best is None:
        _note(i, "no valid target")
        continue
    key = (min(i,best), max(i,best))
    if key in seen:
        continue
    seen.add(key)
    sources.append(i); targets.append(best); straight.append(best_d)
    if len(sources) >= MAX_SAMPLES:
        break

print("sample pairs accepted:", len(sources))
print("\nREJECTIONS BY ZONE (F-8) — shows where sample is being lost")
import pandas as _pd
_rej = _pd.DataFrame([{"zone": z, "reason": r, "count": c} for (z, r), c in rej.items()])
if not _rej.empty:
    display(_rej.pivot_table(index="reason", columns="zone", values="count",
                             aggfunc="sum", fill_value=0))

## 8. Tests — run before any number is reported

Three cases with answers known by construction, including the flyover case that v0.4
got wrong.

In [ ]:
from scipy.sparse.csgraph import dijkstra as _dij

# --- maths guards -------------------------------------------------------------
assert detour_index(1000.0, 50.0) == 20.0
assert detour_index(100.0, 0) is None and detour_index(float("inf"), 50.0) is None
assert oneway_dirs({"oneway":"yes"}) == (True, False)
assert oneway_dirs({"oneway":"-1"}) == (False, True)
assert abs(haversine_m(78.4, 17.4, 78.4, 17.4009) - 100) < 3
assert abs(bearing_deg(78.4, 17.4, 78.4, 17.41) - 0) < 1      # due north
assert abs(bearing_deg(78.4, 17.4, 78.41, 17.4) - 90) < 1     # due east
assert angle_between(10, 350) == 20
assert is_antiparallel(0, 180, 35) and is_antiparallel(0, 200, 35)
assert not is_antiparallel(0, 90, 35), "perpendicular roads must be rejected"
assert not is_antiparallel(0, 10, 35), "parallel same-direction must be rejected"

# --- F-1 level logic ----------------------------------------------------------
assert way_level({}) == 0
assert way_level({"bridge":"yes"}) == 1
assert way_level({"tunnel":"yes"}) == -1
assert way_level({"layer":"2","bridge":"yes"}) == 2
assert way_level({"bridge":"no"}) == 0
assert levels_compatible({0}, {0})
assert not levels_compatible({0}, {1}), "a flyover 15 m overhead is NOT across the road"
print("PASS — bearing and level logic (the v0.4 bugs)")

t, s = strip_military({"highway":"service","landuse":"military","name":"Depot"})
assert s and "landuse" not in t and "name" not in t

# --- synthetic divided road, answer known by hand -----------------------------
M = 21
er, ec, ev = [], [], []
for i in range(M-1):
    er.append(i);     ec.append(i+1);   ev.append(100.0)
for i in range(M-1):
    er.append(M+i+1); ec.append(M+i);   ev.append(100.0)
er.append(M-1); ec.append(2*M-1); ev.append(30.0)      # only crossing, far end
Gt = csr_matrix((ev,(er,ec)), shape=(2*M,2*M))
mid, opp = M//2, M + M//2
dist = _dij(Gt, indices=mid, directed=True, limit=20000)[opp]
di = detour_index(dist, 30.0)
assert abs(dist - 2030.0) < 1.0, f"expected 2030 m, got {dist}"
assert 65 < di < 70, f"expected DI ~67.7, got {di}"
print(f"PASS — synthetic divided road: {dist:.0f} m to cross 30 m, DI {di:.1f}")

# --- the v0.4 failure, as a regression test -----------------------------------
# A surface road and a flyover directly above it, 15 m apart in plan.
surface_levels, flyover_levels = {0}, {1}
assert not levels_compatible(surface_levels, flyover_levels)
# Two unrelated streets crossing at 90 degrees, 20 m apart at the corner.
assert not is_antiparallel(45, 135, BEARING_TOL)
print("PASS — v0.4 artefacts are now rejected")
print("PASS — all tests")

## 9. Route every accepted pair

In [ ]:
import time
from scipy.sparse.csgraph import dijkstra

t0 = time.time()
net = np.empty(len(sources))
CHUNK = 200
for s in range(0, len(sources), CHUNK):
    idxs = sources[s:s+CHUNK]
    D = dijkstra(G, directed=True, indices=idxs, limit=ROUTE_LIMIT_M)
    for r, t in enumerate(targets[s:s+CHUNK]):
        net[s+r] = D[r, t]
    print(f"  routed {min(s+CHUNK,len(sources))}/{len(sources)} ({time.time()-t0:.0f}s)", end="\r")
print()
print(f"routing done in {time.time()-t0:.0f}s")

## 10. Results — uncensored headline (F-6)

The headline distribution below includes **every routed point**. Points above the review
threshold are marked, not removed. The censored view from v0.5 is shown underneath for
comparison, so the difference the censoring made is visible rather than hidden.

In [ ]:
import pandas as pd

recs = []
for i,(si,ti,st) in enumerate(zip(sources,targets,straight)):
    di = detour_index(net[i], st)
    zs = zone_of(lon[si], lat[si], ZONES)
    recs.append({
        "lat": round(float(lat[si]),6), "lon": round(float(lon[si]),6),
        "straight_m": round(float(st),1),
        "network_m": None if not np.isfinite(net[i]) else round(float(net[i]),1),
        "detour_index": None if di is None else round(float(di),2),
        "unreachable": not np.isfinite(net[i]),
        "near_limit": bool(np.isfinite(net[i]) and net[i] > 0.9*ROUTE_LIMIT_M),
        "flagged_high": bool(di is not None and di >= DI_REVIEW),
        "diff_component": bool(comp_label[si] != comp_label[ti]),
        "core_city": "core_city" in zs, "patancheru": "patancheru" in zs,
    })
res = pd.DataFrame(recs)
# F-15: a pair whose endpoints sit in different graph components has no route in our
# data at all. That is a data gap, not a detour, so it is reported separately rather
# than counted as an enormous detour (the v0.6 inflation).
gap = res[res.diff_component]
routed = res[res.detour_index.notna() & (~res.diff_component)]
censored = routed[(~routed.near_limit) & (~routed.flagged_high)]   # the v0.5 view

print("DATA QUALITY")
print(f"  pairs attempted     : {len(res):,}")
print(f"  routed              : {len(routed):,}")
print(f"  unreachable         : {int(res.unreachable.sum()):,}")
print(f"  near routing limit  : {int(res.near_limit.sum()):,}")
print(f"  flagged DI >= {DI_REVIEW:.0f}   : {int(res.flagged_high.sum()):,} "
      f"({100*res.flagged_high.mean():.1f}% of all pairs)")
print(f"  DIFFERENT COMPONENT : {int(res.diff_component.sum()):,} "
      f"<- no route EXISTS in our data; a gap, not a detour")
print()
print("=" * 62)
print("HEADLINE — ALL ROUTED POINTS, NOTHING REMOVED (F-6)")
print("=" * 62)
print(routed.detour_index.describe(percentiles=[.25,.5,.75,.9,.95,.99]).round(2).to_string())
print()
print("For comparison, the v0.5 censored view (worst points deleted):")
print(censored.detour_index.describe(percentiles=[.5,.9]).round(2).to_string())
print()
print(f"CENSORING EFFECT: median {censored.detour_index.median():.2f} (censored) "
      f"vs {routed.detour_index.median():.2f} (true). "
      f"Report the second number.")

In [ ]:
def dedupe_worst(df, n=15, radius_m=CLUSTER_M):
    kept = []
    for _, r in df.sort_values("detour_index", ascending=False).iterrows():
        if all(haversine_m(r.lon, r.lat, k["lon"], k["lat"]) > radius_m for k in kept):
            kept.append({"lat": r.lat, "lon": r.lon, "straight_m": r.straight_m,
                         "network_m": r.network_m, "detour_index": r.detour_index,
                         "near_limit": r.near_limit, "patancheru": r.patancheru})
        if len(kept) >= n:
            break
    return pd.DataFrame(kept)
print("ready")

## 10a. The citywide area atlas

Every sample point is assigned to the nearest named place. Areas are ranked by median
Detour Index. This ranking is the output of the project so far.

In [ ]:
from scipy.spatial import cKDTree as _KD

place_names = [p[0] for p in g.places]
place_kind  = {p[0]: p[1] for p in g.places}
pxy = np.array([[p[2]*(111320.0*math.cos(math.radians(LAT0))), p[3]*110540.0]
                for p in g.places]) if g.places else np.zeros((0,2))

if len(pxy):
    ptree = _KD(pxy)
    qxy = np.column_stack([routed.lon.values*(111320.0*math.cos(math.radians(LAT0))),
                           routed.lat.values*110540.0])
    dist, idx = ptree.query(qxy)
    routed = routed.copy()
    routed["area"] = [place_names[i] if d <= PLACE_MAX_M else None
                      for d, i in zip(dist, idx)]
    routed["area_dist_m"] = np.round(dist, 0)
else:
    routed = routed.copy(); routed["area"] = None

assigned = routed[routed.area.notna()]
print(f"points assigned to a named area: {len(assigned):,} of {len(routed):,}")
print(f"distinct areas                 : {assigned.area.nunique():,}")

atlas = (assigned.groupby("area")
         .agg(points=("detour_index","size"),
              median_DI=("detour_index","median"),
              p90_DI=("detour_index", lambda s: s.quantile(0.9)),
              pct_severe=("detour_index", lambda s: 100*(s>=DI_SEVERE).mean()),
              pct_flagged=("flagged_high","mean"),
              median_straight_m=("straight_m","median"),
              median_network_m=("network_m","median"))
         .reset_index())
atlas["pct_flagged"] = (100*atlas.pct_flagged).round(1)
atlas["kind"] = atlas.area.map(place_kind)
atlas["confidence"] = np.where(atlas.points >= MIN_AREA_N, "ok", "low (small n)")
for c in ["median_DI","p90_DI","pct_severe","median_straight_m","median_network_m"]:
    atlas[c] = atlas[c].round(2)
atlas = atlas.sort_values("median_DI", ascending=False).reset_index(drop=True)

solid = atlas[atlas.confidence == "ok"]
print(f"areas with n >= {MIN_AREA_N}: {len(solid):,}")

## 10b. The ring — nineteen sectors, 360 degrees

Each measured point is placed by its BEARING from the city centre, so every sector is a
wedge of the city and the nineteen together leave no gap. Distance from the centre puts
it in the core, the ORR band, or beyond.

In [ ]:
def bearing_from_city(lon_, lat_):
    return bearing_deg(CITY[0], CITY[1], lon_, lat_)

def km_from_city(lon_, lat_):
    return haversine_m(CITY[0], CITY[1], lon_, lat_) / 1000.0

_names = [e[0] for e in ORR_EXITS]
_brgs  = [e[1] for e in ORR_EXITS]

def sector_of(b):
    """Nearest exit bearing, wrapping correctly at 360."""
    best, bd = None, 999
    for nm, eb in ORR_EXITS:
        d = abs(((b - eb + 180) % 360) - 180)
        if d < bd:
            best, bd = nm, d
    return best

def band_of(km):
    if km <= ORR_INNER_KM: return "core (inside ORR)"
    if km <= ORR_BAND_KM:  return "ORR band"
    return "beyond the ORR"

routed = routed.copy()
routed["bearing"] = [bearing_from_city(lo, la) for lo, la in zip(routed.lon, routed.lat)]
routed["km_out"]  = [round(km_from_city(lo, la), 2) for lo, la in zip(routed.lon, routed.lat)]
routed["sector"]  = [sector_of(b) for b in routed.bearing]
routed["band"]    = [band_of(k) for k in routed.km_out]

print("points per band:")
print(routed.band.value_counts().to_string())
print()
print(f"sectors with any data: {routed.sector.nunique()} of {len(ORR_EXITS)}")

In [ ]:
sect = (routed.groupby("sector")
        .agg(points=("detour_index","size"),
             median_DI=("detour_index","median"),
             p90_DI=("detour_index", lambda s: s.quantile(0.9)),
             pct_severe=("detour_index", lambda s: 100*(s>=DI_SEVERE).mean()),
             pct_flagged=("flagged_high","mean"),
             median_network_m=("network_m","median"),
             median_km_out=("km_out","median"))
        .reset_index())
sect["pct_flagged"] = (100*sect.pct_flagged).round(1)
sect["bearing"] = sect.sector.map(dict(ORR_EXITS))
sect["confidence"] = np.where(sect.points >= MIN_AREA_N, "ok", "low (small n)")
for c in ["median_DI","p90_DI","pct_severe","median_network_m","median_km_out"]:
    sect[c] = sect[c].round(2)
sect = sect.sort_values("median_DI", ascending=False).reset_index(drop=True)

print("ALL NINETEEN SECTORS, WORST FIRST")
display(sect[["sector","bearing","points","median_DI","p90_DI","pct_severe","confidence"]])

ok_s = sect[sect.confidence == "ok"]
if len(ok_s) >= 2:
    w, b = ok_s.iloc[0], ok_s.iloc[-1]
    print()
    print(f"WORST SECTOR : {w.sector:18s} median DI {w.median_DI:7.2f}  ({int(w.points)} pts)")
    print(f"BEST SECTOR  : {b.sector:18s} median DI {b.median_DI:7.2f}  ({int(b.points)} pts)")
    print(f"SPREAD       : {w.median_DI / max(b.median_DI, 0.01):.1f}x across the ring")

## 10c. What are the flagged points? (E-1)

A third of the sample comes out above DI 50. Four explanations are possible and they
predict different things, so each one is tested rather than assumed.

| If the cause is | Then flagged points should |
|---|---|
| Real severance on the periphery | rise with distance from the centre |
| Study-area edge effects | cluster near the boundary of the measured area |
| Missing service roads in OSM | sit on arterials whose frontage has no mapped access |
| A few bad places over-sampled | cluster tightly in space |


In [ ]:
flag = routed[routed.flagged_high]
print(f"flagged: {len(flag):,} of {len(routed):,}  ({100*len(flag)/len(routed):.1f}%)")
print()

# --- test 1: does flagging rise with distance from the centre? -----------------
print("TEST 1 — flagged share by ring band")
t1 = (routed.groupby("band")
      .agg(points=("flagged_high","size"), flagged=("flagged_high","sum"))
      .reset_index())
t1["pct_flagged"] = (100*t1.flagged/t1.points).round(1)
order = {"core (inside ORR)":0, "ORR band":1, "beyond the ORR":2}
t1["_o"] = t1.band.map(order)
display(t1.sort_values("_o").drop(columns="_o").reset_index(drop=True))

### Test 2 — the edge effect

A point near the boundary of the measured area has part of its road network cut away,
so the router cannot find routes that exist in reality. The clip carried a buffer, but
the *measured* area is the HMDA box, and points close to its edge are still suspect.

If flagging rises sharply in the outermost kilometres, the periphery result is partly an
artefact of where we stopped measuring, not of how the roads are built.

In [ ]:
x0, y0, x1, y1 = ZONES["hmda_all"]

def edge_km(lon_, lat_):
    """Straight-line distance to the nearest edge of the measured area."""
    dx = min(abs(lon_ - x0), abs(x1 - lon_)) * 111.320 * math.cos(math.radians(lat_))
    dy = min(abs(lat_ - y0), abs(y1 - lat_)) * 110.540
    return round(min(dx, dy), 2)

routed = routed.copy()
routed["edge_km"] = [edge_km(lo, la) for lo, la in zip(routed.lon, routed.lat)]

bins = [(0,2,"0-2 km from edge"), (2,5,"2-5 km"), (5,10,"5-10 km"),
        (10,20,"10-20 km"), (20,999,"20 km+ (interior)")]
rows2 = []
for lo_, hi_, lab in bins:
    d = routed[(routed.edge_km >= lo_) & (routed.edge_km < hi_)]
    if len(d) == 0:
        continue
    rows2.append({"distance from study edge": lab, "points": len(d),
                  "median_DI": round(d.detour_index.median(), 2),
                  "pct_flagged": round(100*d.flagged_high.mean(), 1)})
edge_tbl = pd.DataFrame(rows2)
print("TEST 2 — is flagging an edge effect?")
display(edge_tbl)

if len(edge_tbl) >= 2:
    near = edge_tbl.iloc[0].pct_flagged
    far = edge_tbl.iloc[-1].pct_flagged
    if near > far * 1.4:
        print()
        print(f"EDGE EFFECT PRESENT: {near}% flagged within 2 km of the boundary vs")
        print(f"  {far}% in the interior. Points near the edge are NOT trustworthy and")
        print("  are excluded from the corrected headline below.")
    else:
        print()
        print(f"NO STRONG EDGE EFFECT: {near}% near the edge vs {far}% interior.")
        print("  The flagged population is not explained by where we stopped measuring.")

### Test 3 — are they a few bad places, or spread across the city?

In [ ]:
# cluster the flagged points at 300 m and see how concentrated they are
CL = 300.0
cl_pts, counts = [], []
for _, r in flag.iterrows():
    placed = False
    for i in range(len(cl_pts)):
        if haversine_m(r.lon, r.lat, cl_pts[i][0], cl_pts[i][1]) <= CL:
            counts[i] += 1; placed = True; break
    if not placed:
        cl_pts.append((r.lon, r.lat)); counts.append(1)

counts.sort(reverse=True)
tot = sum(counts)
print(f"TEST 3 — {len(flag):,} flagged points fall into {len(cl_pts):,} distinct")
print(f"  locations more than {int(CL)} m apart.")
print(f"  points per location: {tot/max(len(cl_pts),1):.1f}")
if len(counts):
    top10 = sum(counts[:10])
    print(f"  the 10 busiest locations hold {top10:,} points ({100*top10/tot:.1f}% of flagged)")
print()
if tot / max(len(cl_pts), 1) < 3:
    print("SPREAD ACROSS THE CITY. This is a pattern, not a handful of bad junctions.")
    print("  That makes it a finding about design, and it also means it cannot be")
    print("  dismissed as a few mapping errors.")
else:
    print("CONCENTRATED. A limited number of places are being sampled repeatedly.")
    print("  Verify the busiest locations before treating the share as citywide.")

### Test 4 — what kind of road are they on?

In [ ]:
# network distance distribution tells us whether these are long real detours
# or routing that wandered the whole city
q = flag.network_m.quantile([0.25, 0.5, 0.75, 0.95]).round(0)
print("TEST 4 — network distance of flagged points")
print(f"  25th percentile : {q.loc[0.25]:>9,.0f} m")
print(f"  median          : {q.loc[0.50]:>9,.0f} m")
print(f"  75th percentile : {q.loc[0.75]:>9,.0f} m")
print(f"  95th percentile : {q.loc[0.95]:>9,.0f} m")
print(f"  straight-line median across the same points: {flag.straight_m.median():.1f} m")
print()
under2 = (flag.network_m < 2000).mean() * 100
over10 = (flag.network_m > 10000).mean() * 100
print(f"  under 2 km  : {under2:.1f}%   plausible as a real detour")
print(f"  over 10 km  : {over10:.1f}%   implausible; OSM is missing a connection")
print()
print("A real median detour is hundreds of metres to a few kilometres. Anything above")
print("about 10 km to cross 20 m is the map missing a link, not the road being built")
print("that way — which is the Section 18 conclusion, now quantified.")

## 10d. Corrected headline (E-1, E-2)

Two corrections applied together: points within 2 km of the study-area boundary are
dropped, and points whose route exceeds 10 km to cross a single carriageway are
separated out as map gaps rather than counted as detours.

In [ ]:
clean = routed[(routed.edge_km >= 2.0) & (routed.network_m <= 10000)]
gaps  = routed[routed.network_m > 10000]

print(f"raw routed      : {len(routed):,}")
print(f"map-gap points  : {len(gaps):,}   (route over 10 km to cross one carriageway)")
print(f"edge-suspect    : {len(routed[routed.edge_km < 2.0]):,}")
print(f"CLEAN SAMPLE    : {len(clean):,}")
print()
print("CORRECTED FIGURES, clean sample only")
rows3 = []
for lab, m in [("all Hyderabad", np.ones(len(clean), bool)),
               ("core (inside ORR)", (clean.band == "core (inside ORR)").values),
               ("ORR band", (clean.band == "ORR band").values),
               ("beyond the ORR", (clean.band == "beyond the ORR").values)]:
    d = clean[m]
    if d.empty:
        continue
    rows3.append({"zone": lab, "points": len(d),
                  "median_DI": round(d.detour_index.median(), 2),
                  "p90_DI": round(d.detour_index.quantile(0.9), 2),
                  "median_network_m": round(d.network_m.median(), 1)})
clean_tbl = pd.DataFrame(rows3)
display(clean_tbl)

if len(clean_tbl) >= 4:
    c = clean_tbl[clean_tbl.zone == "core (inside ORR)"].iloc[0].median_DI
    o = clean_tbl[clean_tbl.zone == "beyond the ORR"].iloc[0].median_DI
    print()
    print(f"CORRECTED GRADIENT: periphery is {o/c:.2f}x the core.")
    print("  Compare with the uncorrected v0.9 figure of 2.85x. If these are close,")
    print("  the gradient survives both corrections and is the finding to publish.")

## 10e. Sectors compared within ring bands (E-2, E-3)

v0.9's sectors sampled at depths from 7.6 km to 23.4 km. Comparing each sector only
against others in the same ring band removes that difference, so no sector can score
badly merely for sampling further out.

In [ ]:
strat = (clean.groupby(["band","sector"])
         .agg(points=("detour_index","size"),
              median_DI=("detour_index","median"))
         .reset_index())
strat["median_DI"] = strat.median_DI.round(2)
strat["adequate"] = np.where(strat.points >= MIN_AREA_N, "yes", "no (small n)")

for b in ["core (inside ORR)", "ORR band", "beyond the ORR"]:
    sub = strat[(strat.band == b) & (strat.adequate == "yes")].sort_values("median_DI", ascending=False)
    if sub.empty:
        continue
    print(f"\n{b.upper()} — sectors with n >= {MIN_AREA_N}, worst first")
    display(sub[["sector","points","median_DI"]].reset_index(drop=True))
    if len(sub) >= 2:
        print(f"  spread within this band: {sub.iloc[0].median_DI / max(sub.iloc[-1].median_DI, 0.01):.1f}x")

thin = strat[strat.adequate != "yes"]
if len(thin):
    print(f"\nEXCLUDED for small sample: {len(thin)} sector-band cells")
    display(thin[["band","sector","points"]].reset_index(drop=True))

In [ ]:
band = (routed.groupby("band")
        .agg(points=("detour_index","size"),
             median_DI=("detour_index","median"),
             p90_DI=("detour_index", lambda s: s.quantile(0.9)),
             pct_severe=("detour_index", lambda s: 100*(s>=DI_SEVERE).mean()),
             median_network_m=("network_m","median"))
        .reset_index())
for c in ["median_DI","p90_DI","pct_severe","median_network_m"]:
    band[c] = band[c].round(2)
order = {"core (inside ORR)": 0, "ORR band": 1, "beyond the ORR": 2}
band["_o"] = band.band.map(order)
band = band.sort_values("_o").drop(columns="_o").reset_index(drop=True)

print("DOES SEVERANCE RISE WITH DISTANCE FROM THE CENTRE?")
display(band)

if len(band) == 3:
    inner, outer = band.iloc[0].median_DI, band.iloc[2].median_DI
    print()
    if outer > inner * 1.15:
        print(f"YES — beyond the ORR is {outer/inner:.1f}x the core. The periphery is")
        print("      measurably worse served than the centre.")
    elif inner > outer * 1.15:
        print(f"NO — the core is worse ({inner:.1f} vs {outer:.1f}). Likely the old city's")
        print("      genuine one-way streets inflating the core. Investigate before quoting.")
    else:
        print(f"NO CLEAR DIFFERENCE ({inner:.1f} core vs {outer:.1f} outer). The periphery")
        print("      hypothesis is NOT supported by this measurement.")

In [ ]:
# A compass printout: the whole city at a glance, in reading order round the ring.
print("THE RING, CLOCKWISE FROM NORTH")
print(f"{'sector':20s} {'brg':>5s} {'n':>6s} {'median':>8s}  severity")
for nm, br in sorted(ORR_EXITS, key=lambda e: e[1]):
    r = sect[sect.sector == nm]
    if not len(r):
        print(f"{nm:20s} {br:5d} {'-':>6s} {'no data':>8s}")
        continue
    r = r.iloc[0]
    bar = "#" * int(min(40, r.median_DI / 1.5))
    flag = "" if r.confidence == "ok" else "  (low n)"
    print(f"{nm:20s} {br:5d} {int(r.points):6d} {r.median_DI:8.2f}  {bar}{flag}")

In [ ]:
print("WORST 25 AREAS IN HYDERABAD (n >= " + str(MIN_AREA_N) + ")")
display(solid.head(25)[["area","kind","points","median_DI","p90_DI","pct_severe"]])

print("\nBEST 15 AREAS — least severed")
display(solid.tail(15)[["area","kind","points","median_DI","pct_severe"]]
        .sort_values("median_DI"))

print("\nCITY CONTEXT")
print(f"  citywide median DI           : {routed.detour_index.median():.2f}")
print(f"  best area median             : {solid.median_DI.min():.2f}  ({solid.loc[solid.median_DI.idxmin(),'area']})")
print(f"  worst area median            : {solid.median_DI.max():.2f}  ({solid.loc[solid.median_DI.idxmax(),'area']})")
print(f"  spread, worst / best         : {solid.median_DI.max()/max(solid.median_DI.min(),0.01):.1f}x")

In [ ]:
# Where does the pilot corridor sit in the citywide ranking?
for target in ["Isnapur", "Patancheru", "Muthangi", "Rudraram", "Chitkul"]:
    row = solid[solid.area.str.lower() == target.lower()]
    if len(row):
        r = row.iloc[0]
        rank = int(solid.median_DI.rank(ascending=False, method="min")[row.index[0]])
        print(f"{target:14s} rank {rank:3d} of {len(solid)}  median DI {r.median_DI:7.2f}  n={int(r.points)}")
    else:
        print(f"{target:14s} not in the ranking (too few points or not named in OSM)")

In [ ]:
rows = []
for label, mask in [("all Hyderabad", np.ones(len(routed),bool)),
                    ("core city", routed.core_city.values),
                    ("patancheru", routed.patancheru.values)]:
    d = routed[mask]
    if d.empty:
        continue
    rows.append({"zone": label, "points": len(d),
                 "median_DI": round(d.detour_index.median(),2),
                 "p90_DI": round(d.detour_index.quantile(0.9),2),
                 "pct_flagged": round(100*d.flagged_high.mean(),1)})
zone_tbl = pd.DataFrame(rows)
display(zone_tbl)

worst = dedupe_worst(routed, 15, CLUSTER_M)

## 10b. Verification sample (F-10)

The middle of the distribution looks plausible — roughly 300 m of driving to cross a
divided arterial. Plausible is an argument, not evidence. These are ordinary mid-range
points, picked at random with a fixed seed so the selection cannot be tuned. Open a few
on a map and check whether the claimed detour is real.

In [ ]:
mid = routed[(routed.detour_index >= 8) & (routed.detour_index <= 16)]
if len(mid):
    sample = mid.sample(min(10, len(mid)), random_state=42)[
        ["lat","lon","straight_m","network_m","detour_index","patancheru"]]
    display(sample)
    print("\nPaste any lat,lon into a map. Question for each: is there really no way")
    print("across within about " + str(int(sample.straight_m.median())) + " m of that point?")
else:
    sample = routed.head(0)
    print("No mid-range points found.")

## 11. Validation route — your own trip

Put in the two endpoints you actually drive. The graph gives the shortest legal driving
distance by pure geometry, with no congestion and no construction in it. If that number
matches what you know the drive to be, the graph is sound and your 54 minutes is
traffic. If it is wildly longer, the graph is still broken.

In [ ]:
# Fill these in: (name, lat, lon). Leave empty to skip.
ROUTE_CHECKS = [
    # ("Isnapur cheruvu", 17.5xxx, 78.2xxx),
    # ("Janapriya Unnati", 17.5xxx, 78.3xxx),
]

if len(ROUTE_CHECKS) >= 2:
    from scipy.sparse.csgraph import dijkstra as _d
    def nearest_node(latq, lonq):
        px = lonq * (111320.0*math.cos(math.radians(LAT0)))
        py = latq * 110540.0
        return int(tree.query([px, py])[1])

    for a in range(len(ROUTE_CHECKS)-1):
        n1, la1, lo1 = ROUTE_CHECKS[a]
        n2, la2, lo2 = ROUTE_CHECKS[a+1]
        i1, i2 = nearest_node(la1, lo1), nearest_node(la2, lo2)
        d = _d(G, directed=True, indices=i1, limit=100000)[i2]
        sl = haversine_m(lo1, la1, lo2, la2)
        print(f"{n1} -> {n2}")
        print(f"  straight line   : {sl/1000:.2f} km")
        print(f"  shortest driving: {'unreachable' if not np.isfinite(d) else f'{d/1000:.2f} km'}")
        if np.isfinite(d):
            print(f"  detour index    : {d/sl:.2f}")
else:
    print("No route check configured — add two coordinates above and re-run this cell.")

## 12. What these numbers are and are not

- They measure **distance and geometry**, never travel time. Congestion and
  construction are deliberately absent. That is what makes the claim survive the
  roadworks finishing.
- Every high-DI point is a **candidate to verify**, not a finding. OSM gaps invent
  detours; a missing service-road link looks exactly like a severed neighbourhood.
- The **flagged** count in §10 is the honest measure of how much the method still
  cannot explain. It marks points; it never removes them from the headline.
- Compare the rejection counts in §7 against v0.4's results. The size of the
  "grade separated" and "not antiparallel" buckets tells you how much of v0.4's
  alarming output was an artefact of my own code.

## 13. Save outputs and push to Drive

In [ ]:
import datetime, shutil
stamp = datetime.date.today().isoformat()

res.to_csv(f"{OUT_DIR}/di_v10_points_{stamp}.csv", index=False)
zone_tbl.to_csv(f"{OUT_DIR}/di_v10_by_zone_{stamp}.csv", index=False)
atlas.to_csv(f"{OUT_DIR}/AREA_ATLAS_{stamp}.csv", index=False)
sect.to_csv(f"{OUT_DIR}/ORR_SECTORS_{stamp}.csv", index=False)
band.to_csv(f"{OUT_DIR}/RING_BANDS_{stamp}.csv", index=False)
edge_tbl.to_csv(f"{OUT_DIR}/EDGE_TEST_{stamp}.csv", index=False)
clean_tbl.to_csv(f"{OUT_DIR}/CORRECTED_BANDS_{stamp}.csv", index=False)
strat.to_csv(f"{OUT_DIR}/SECTORS_STRATIFIED_{stamp}.csv", index=False)
sample.to_csv(f"{OUT_DIR}/di_v10_verification_sample_{stamp}.csv", index=False)
gap.to_csv(f"{OUT_DIR}/di_v10_data_gaps_{stamp}.csv", index=False)
worst.to_csv(f"{OUT_DIR}/di_v10_worst_{stamp}.csv", index=False)
_rej.to_csv(f"{OUT_DIR}/di_v10_rejections_by_zone_{stamp}.csv", index=False)

with open(f"{OUT_DIR}/RING_ANALYSIS_NOTES_{stamp}.md","w") as f:
    f.write(f"# arcroad Phase 2e v1.0 — {stamp}\n\n")
    f.write(f"Graph {N:,} nodes / {G.nnz:,} edges. Pairs {len(res):,}, "
            f"routed {len(routed):,}, unreachable {int(res.unreachable.sum()):,}, "
            f"near-limit {int(res.near_limit.sum()):,}, "
            f"flagged DI>={DI_REVIEW:.0f} {int(res.flagged_high.sum()):,}.\n\n")
    f.write(f"HEADLINE median DI (uncensored): {routed.detour_index.median():.2f}. "
            f"Censored (v0.5 method) would have said {censored.detour_index.median():.2f}.\n\n")
    f.write("Map data (c) OpenStreetMap contributors, ODbL. "
            "HMA derived metrics under ODbL. Distance/geometry only — not travel time.\n\n")
    f.write(zone_tbl.to_markdown(index=False))
    f.write("\n\n## Rejections by zone\n\n")
    f.write(_rej.to_markdown(index=False))
    f.write("\n\n## ORR sectors, worst first\n\n")
    f.write(sect[["sector","bearing","points","median_DI","pct_severe","confidence"]].to_markdown(index=False))
    f.write("\n\n## Ring bands\n\n")
    f.write(band.to_markdown(index=False))
    f.write("\n\n## Worst 25 areas\n\n")
    f.write(solid.head(25)[["area","kind","points","median_DI","p90_DI","pct_severe"]].to_markdown(index=False))
    f.write("\n\n## Worst 15 points, uncensored (candidates, not findings)\n\n")
    f.write(worst.to_markdown(index=False))

from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Anthropic/Hyderabad Mobility Atlas/phase2e_diagnostics"
os.makedirs(dest, exist_ok=True)
for fn in os.listdir(OUT_DIR):
    shutil.copy(f"{OUT_DIR}/{fn}", f"{dest}/{fn}")
print("saved to Drive:", dest)
print(sorted(os.listdir(dest)))